In [2]:
import numpy as np
import astropy.units as u
from astropy.io import fits
import time
from datetime import datetime
today = int(datetime.today().strftime('%Y%m%d'))
from skimage.registration import phase_cross_correlation
import copy
from importlib import reload
from IPython.display import clear_output, display
from pathlib import Path
import os
cwd = Path(os.getcwd())

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize, CenteredNorm
from matplotlib.patches import Circle

import magpyx
from magpyx.utils import ImageStream
import purepyindi
from purepyindi import INDIClient
import purepyindi2
from purepyindi2 import IndiClient

client0 = INDIClient('localhost', 7624)
client0.start()
client = IndiClient()
client.connect()
client.get_properties()

import lina
from lina.math_module import xp, xcipy, ensure_np_array
from lina import utils, rt_utils, coro_utils

import fsm_utils
import cam_utils

v_bias = np.array([[50,50,50]]).T
zero = np.array([[0,0,0]]).T

wavelength = 633e-9
fl = 500e-3
fsm_pupil_diam = 6.7e-3
rad_per_lamD = wavelength/fsm_pupil_diam
print(rad_per_lamD)
as_per_lamD = (wavelength/fsm_pupil_diam*u.radian).to_value(u.arcsec)
print(as_per_lamD)

pxscl_lamD = 3.45e-6 / (fl * wavelength/fsm_pupil_diam)
print(pxscl_lamD)

INFO:purepyindi2.transports:Opening localhost:7624...


9.44776119402985e-05
19.48740632155403
0.07303317535545024


INFO:purepyindi2.transports:Connected to localhost:7624


In [4]:
camfsm_telem_path = Path(f'/opt/MagAOX/rawimages/camfsm/camfsm/2026_09_25')
camfsm_data_path = Path(f'camfsm-data-{today}')
utils.make_dir(camfsm_data_path)


dmt_telem_path = Path(f'/opt/MagAOX/rawimages/dm00disp/dm00disp/2026_10_01')
dmt_data_path = Path(f'dmt-data-{today}')
utils.make_dir(dmt_data_path)


dm01_telem_path = Path(f'/opt/MagAOX/rawimages/dm00disp01/dm00disp01/2026_10_01')
dm01_data_path = Path(f'dm01-data-{today}')
utils.make_dir(dm01_data_path)



Directory 'camfsm-data-20261001' already exists.
Directory 'dmt-data-20261001' already exists.
Directory 'dm01-data-20261001' already exists.


In [5]:
dmt_channel = 'dm00disp'
dm00_channel = 'dm00disp00'
dm01_channel = 'dm00disp01'

DMT_STREAM = ImageStream(dmt_channel)
DM00_STREAM = ImageStream(dm00_channel)
DM01_STREAM = ImageStream(dm01_channel)

zeros = np.array([[0, 0, 0]]).T
vals = np.array([[1, 0, 0]]).T
DM00_STREAM.write(zeros)

In [6]:
reload(rt_utils)

rt_utils.toggle_telem(1, DMT_STREAM.name, client0)
rt_utils.toggle_telem(1, DM01_STREAM.name, client0)

time.sleep(2)
for i in range(5):
    DM01_STREAM.write(zeros)
    time.sleep(0.5)

rt_utils.toggle_telem(0, DMT_STREAM.name, client0)
rt_utils.toggle_telem(0, DM01_STREAM.name, client0)

In [7]:
reload(rt_utils)
rt_utils.unpack_data(
    dm01_telem_path,
    dm01_data_path,
)

In [8]:
reload(rt_utils)
rt_utils.unpack_data(
    dmt_telem_path,
    dmt_data_path,
)

In [9]:
# fnames = utils.get_fnames(dm01_telem_path/'fits/*.fits')
dmt_fnames = utils.get_fnames(dmt_data_path/'*.fits')
dm1_fnames = utils.get_fnames(dm01_data_path/'*.fits')


In [10]:
reload(rt_utils)
dmt_data, dmt_times = rt_utils.read_telem_data(dmt_fnames, rel_times=0)
dm1_data, dm1_times = rt_utils.read_telem_data(dm1_fnames, rel_times=0)

In [11]:
dmt_times, dm1_times

(array([73866.51901446, 73867.0196126 , 73867.52028053, 73868.02094114,
        73868.52161711]),
 array([73866.51903819, 73867.01965741, 73867.52032998, 73868.0210163 ,
        73868.52169176]))

In [12]:
dmt_times[0] - dm1_times[0]

-2.373200550209731e-05

In [17]:
reload(utils)
utils.move_files(dmt_telem_path, dmt_data_path)

['dm00disp_20261001203106519014457.xrif']
Moved files from /opt/MagAOX/rawimages/dm00disp/dm00disp/2026_10_01 to dmt-data-20261001


In [14]:
utils.delete_files(dm01_telem_path/'*.xrif')

In [9]:
rt_utils.unpack_data(
    camfsm_telem_path,
    camfsm_data_path,
)